pip installations

In [ ]:
# @title
# %pip install -Uq "unstructured[all-docs]"
%pip install -Uq langchain_chroma
# %pip install -Uq langchain langchain-community
# %pip install -Uq python_dotenv
# !pip install docx
!pip uninstall -y langchain-huggingface langchain-core
!pip install "langchain-huggingface==0.3.1" "langchain-core==0.3.72"
!pip install "ragas==0.3.9" "langchain-community==0.3.27"
!pip install sentence-transformers
!pip install rank_bm25

Check RAM

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
# @title
import psutil

ram = psutil.virtual_memory()

print(f"Total RAM     : {ram.total / (1024**3):.2f} GB")
print(f"Used RAM      : {ram.used / (1024**3):.2f} GB")
print(f"Available RAM : {ram.available / (1024**3):.2f} GB")
print(f"RAM Usage     : {ram.percent:.1f}%")

Imports

In [ ]:
# @title
import json
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import os
from openai import OpenAI

from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage

Creating Elements of PDF using Unstructured for all the 10 Papers

In [ ]:
# @title
import zipfile
import os
from unstructured.partition.pdf import partition_pdf


def partition_documents_from_zip(zip_path: str):
    """Extract and partition all PDFs from a ZIP file."""

    all_elements = []

    with zipfile.ZipFile(zip_path, "r") as zip_ref:

        # Get all PDF files inside the ZIP
        pdf_files = [
            file_name
            for file_name in zip_ref.namelist()
            if file_name.lower().endswith(".pdf")
        ]

        print(f"📚 Found {len(pdf_files)} PDF files")

        for i, pdf_file in enumerate(pdf_files, start=1):

            print(f"\n{'=' * 60}")
            print(f"📄 Processing Paper {i}/{len(pdf_files)}: {pdf_file}")
            print(f"{'=' * 60}")

            # Extract PDF from ZIP to temporary location
            zip_ref.extract(pdf_file, "/tmp")

            extracted_path = os.path.join("/tmp", pdf_file)

            # Partition this PDF
            elements = partition_pdf(
                filename=extracted_path,
                strategy="hi_res",
                infer_table_structure=True,
                extract_image_block_types=["Image"],
                extract_image_block_to_payload=True
            )

            print(f"✅ Extracted {len(elements)} elements")

            # Add all elements from this paper to the master list
            all_elements.extend(elements)

    print(f"\n{'=' * 60}")
    print(f"✅ Total elements from all papers: {len(all_elements)}")
    print(f"{'=' * 60}")

    return all_elements


# ---------------------------------------------------------
# Run
# ---------------------------------------------------------

zip_path = "/content/rag_dataset_small.zip"
elements = partition_documents_from_zip(zip_path)

In [ ]:
elements

In [ ]:
# All types of different atomic elements we see from unstructured
set([str(type(el)) for el in elements])

In [ ]:
elements[36].to_dict()

In [ ]:
# @title
# Gather all images
images = [element for element in elements if element.category == 'Image']
print(f"Found {len(images)} images")

images[0].to_dict()
# Use https://codebeautify.org/base64-to-image-converter to view the base64 text

In [ ]:
# @title
# Gather all table
tables = [element for element in elements if element.category == 'Table']
print(f"Found {len(tables)} tables")

tables[0].to_dict()

# Use https://jsfiddle.net/ to view the table html


Chunking by Title

In [ ]:
# @title
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")

    chunks = chunk_by_title(
        elements, # The parsed PDF elements from previous step
        max_characters=3000, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=2400, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )

    print(f"✅ Created {len(chunks)} chunks")
    return chunks

# Create chunks
chunks = create_chunks_by_title(elements)

Take from chunks to dict with seperate dtypes

In [ ]:
# @title
def separate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }

    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__

            # Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)

            # Handle images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)

    content_data['types'] = list(set(content_data['types']))
    return content_data

Summarising Function

In [ ]:
# @title
# def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
#     """Create AI-enhanced summary for mixed content"""

#     try:
#         # Initialize LLM (needs vision model for images)

#         # llm = pipeline(
#         #     "image-text-to-text",
#         #     model="Qwen/Qwen2.5-VL-3B-Instruct",
#         #     device_map="auto"
#         # )

#         # Build the text prompt
#         prompt_text = f"""You are creating a searchable description for document content retrieval.

#         CONTENT TO ANALYZE:
#         TEXT CONTENT:
#         {text}

#         """

#         # Add tables if present
#         if tables:
#             prompt_text += "TABLES:\n"
#             for i, table in enumerate(tables):
#                 prompt_text += f"Table {i+1}:\n{table}\n\n"

#                 prompt_text += """
#                 YOUR TASK:
#                 Generate a comprehensive, searchable description that covers:

#                 1. Key facts, numbers, and data points from text and tables
#                 2. Main topics and concepts discussed
#                 3. Questions this content could answer
#                 4. Visual content analysis (charts, diagrams, patterns in images)
#                 5. Alternative search terms users might use

#                 Make it detailed and searchable - prioritize findability over brevity.

#                 SEARCHABLE DESCRIPTION:"""

#         # Build message content starting with text
#         message_content = [{"type": "text", "text": prompt_text}]

#         # Add images to the message
#         for image_base64 in images:
#             message_content.append({
#                 "type": "image_url",
#                 "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
#             })

#         # Send to AI and get response
#         # response = llm(
#         #     message_content,
#         #     max_new_tokens=500,
#         #     do_sample=False
#         # )
#         response = client.chat.completions.create(
#             model="google/gemini-2.5-flash",
#             messages=[
#                 {
#                     "role": "user",
#                     "content": message_content
#                 }
#             ],
#             max_tokens=500
#         )

#         return response.choices[0].message.content

#     except Exception as e:
#         print(f"     ❌ AI summary failed: {e}")
#         # Fallback to simple summary
#         summary = f"{text[:300]}..."
#         if tables:
#             summary += f" [Contains {len(tables)} table(s)]"
#         if images:
#             summary += f" [Contains {len(images)} image(s)]"
#         return summary

from transformers import pipeline
from typing import List


# Initialize the free local vision-language model ONCE
llm = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen2.5-VL-3B-Instruct",
    device_map="auto"
)


def create_ai_enhanced_summary(
    text: str,
    tables: List[str],
    images: List[str]
) -> str:
    """Create AI-enhanced searchable summary for mixed content."""

    # ---------------------------------------------------------
    # 1. Build the text prompt
    # ---------------------------------------------------------

    prompt_text = f"""
You are creating a searchable description for document content retrieval.

CONTENT TO ANALYZE:

TEXT CONTENT:
{text}
"""

    # ---------------------------------------------------------
    # 2. Add tables if present
    # ---------------------------------------------------------

    if tables:
        prompt_text += "\nTABLES:\n"

        for i, table in enumerate(tables):
            prompt_text += f"""
Table {i + 1}:
{table}

"""

    # ---------------------------------------------------------
    # 3. Add instructions
    # ---------------------------------------------------------

    prompt_text += """
YOUR TASK:

Generate a comprehensive, searchable description that covers:

1. Key facts, numbers, and data points from the text and tables
2. Main topics and concepts discussed
3. Questions this content could answer
4. Visual content analysis of charts, diagrams, figures, and patterns in images
5. Alternative search terms users might use

Make it detailed and searchable. Prioritize findability over brevity.

SEARCHABLE DESCRIPTION:
"""

    # ---------------------------------------------------------
    # 4. Build multimodal message
    # ---------------------------------------------------------

    message_content = [
        {
            "type": "text",
            "text": prompt_text
        }
    ]

    # Add images
    for image_base64 in images:
        message_content.append({
            "type": "image",
            "url": f"data:image/jpeg;base64,{image_base64}"
        })

    # ---------------------------------------------------------
    # 5. Generate response using local model
    # ---------------------------------------------------------

    response = llm(
        [
            {
                "role": "user",
                "content": message_content
            }
        ],
        max_new_tokens=500,
        do_sample=False
    )

    # ---------------------------------------------------------
    # 6. Extract generated text
    # ---------------------------------------------------------

    return response[0]["generated_text"][-1]["content"]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

KeyboardInterrupt: 

Memory Efficient creation to Doc dtypeOne

In [ ]:
# @title
import os
import json
import gc
from langchain_core.documents import Document

def summarise_chunks(
    chunks,
    output_dir="dbv1/processed_documents"
):
    """
    Process chunks one at a time and save each resulting
    LangChain Document to disk.

    This avoids keeping all processed Documents in RAM.
    """

    print("🧠 Processing chunks with AI Summaries...")

    # ---------------------------------------------------------
    # Create output directory once
    # ---------------------------------------------------------

    os.makedirs(output_dir, exist_ok=True)

    total_chunks = len(chunks)

    # One JSONL file containing one Document per line
    output_file = os.path.join(
        output_dir,
        "processed_documents.jsonl"
    )

    # ---------------------------------------------------------
    # Process one chunk at a time
    # ---------------------------------------------------------

    with open(output_file, "w", encoding="utf-8") as f:

        for i, chunk in enumerate(chunks):

            current_chunk = i + 1

            print(
                f"\n   Processing chunk "
                f"{current_chunk}/{total_chunks}"
            )

            # -------------------------------------------------
            # 1. Analyze content
            # -------------------------------------------------

            content_data = separate_content_types(chunk)

            print(
                f"     Types found: "
                f"{content_data['types']}"
            )

            print(
                f"     Tables: "
                f"{len(content_data['tables'])}, "
                f"Images: "
                f"{len(content_data['images'])}"
            )

            # -------------------------------------------------
            # 2. Create AI-enhanced content
            # -------------------------------------------------

            if (
                content_data['tables']
                or content_data['images']
            ):

                print(
                    "     → Creating AI summary "
                    "for mixed content..."
                )

                try:

                    enhanced_content = (
                        create_ai_enhanced_summary(
                            content_data['text'],
                            content_data['tables'],
                            content_data['images']
                        )
                    )

                    # Make sure we actually received text
                    if not enhanced_content:
                        raise ValueError(
                            "AI model returned empty summary"
                        )

                    print(
                        "     → AI summary created successfully"
                    )

                    print(
                        f"     → Enhanced content preview: "
                        f"{enhanced_content[:200]}..."
                    )

                except Exception as e:

                    print(
                        f"     ❌ AI summary failed: {e}"
                    )

                    # Safe fallback
                    enhanced_content = (
                        content_data['text']
                    )

            else:

                print(
                    "     → Using raw text "
                    "(no tables/images)"
                )

                enhanced_content = (
                    content_data['text']
                )

            # -------------------------------------------------
            # 3. Create LangChain Document
            # -------------------------------------------------

            doc = Document(
                page_content=enhanced_content,
                metadata={
                    "original_content": json.dumps(
                        {
                            "raw_text":
                                content_data['text'],

                            "tables_html":
                                content_data['tables'],

                            # IMPORTANT:
                            # Keep images because they are
                            # needed later during retrieval.
                            "images_base64":
                                content_data['images']
                        },
                        ensure_ascii=False
                    )
                }
            )

            # -------------------------------------------------
            # 4. Convert Document to JSON-serializable record
            # -------------------------------------------------

            record = {
                "page_content": doc.page_content,
                "metadata": doc.metadata
            }

            # -------------------------------------------------
            # 5. Immediately write to DISK
            # -------------------------------------------------

            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

            # Make sure it is actually written
            f.flush()

            print("     💾 Document saved to disk")

            # -------------------------------------------------
            # 6. Free RAM used by this chunk
            # -------------------------------------------------

            del doc
            del record
            del content_data
            del enhanced_content

            gc.collect()

    print("\n" + "=" * 60)
    print("✅ All chunks processed")
    print(f"📁 Saved to: {output_file}")
    print("=" * 60)

    return output_file

In [ ]:
processed_file = summarise_chunks(chunks)

Create a Vector database (normal way)

In [ ]:
# @title
def create_vector_store(documents, persist_directory="dbv1/chroma_db"):
    """Create and persist ChromaDB vector store"""
    print("🔮 Creating embeddings and storing in ChromaDB...")

    embedding_model = SentenceTransformer(
                      "BAAI/bge-base-en-v1.5"
                      )

    # Create ChromaDB vector store
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("--- Finished creating vector store ---")

    print(f"✅ Vector store created and saved to {persist_directory}")
    return vectorstore

# Create the vector store
db = create_vector_store(processed_chunks)

Helper for loading the disk- cached docs

In [ ]:
# @title
def load_documents_in_batches(
    file_path,
    batch_size=25
):
    """
    Read saved Documents from disk in batches.

    Only one batch is held in RAM at a time.
    """

    batch = []

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            record = json.loads(line)

            doc = Document(
                page_content=record["page_content"],
                metadata=record["metadata"]
            )

            batch.append(doc)

            # When batch is full
            if len(batch) >= batch_size:

                yield batch

                # Free batch
                batch = []

                gc.collect()

        # Return final smaller batch
        if batch:
            yield batch

Create Vectordb efficient way

In [ ]:
# @title
from sentence_transformers import SentenceTransformer
from langchain_chroma import Chroma


def create_vector_store(
    processed_file,
    persist_directory="dbv1/chroma_db",
    batch_size=25
):
    """
    Create ChromaDB from Documents stored on disk.

    Documents are loaded and embedded in batches
    to reduce RAM usage.
    """

    print(
        "🔮 Creating embeddings and storing "
        "in ChromaDB..."
    )

    # ---------------------------------------------------------
    # 1. Load embedding model ONCE
    # ---------------------------------------------------------

    print("📦 Loading embedding model...")

    embedding_model = SentenceTransformer(
        "BAAI/bge-base-en-v1.5"
    )

    print("✅ Embedding model loaded")

    # ---------------------------------------------------------
    # 2. Create ChromaDB using first batch
    # ---------------------------------------------------------

    vectorstore = None

    total_added = 0

    for batch_number, batch in enumerate(
        load_documents_in_batches(
            processed_file,
            batch_size=batch_size
        ),
        start=1
    ):

        print(
            f"\n🔄 Processing batch "
            f"{batch_number}"
        )

        print(
            f"   Documents in batch: "
            f"{len(batch)}"
        )

        # -----------------------------------------------------
        # First batch → create vector store
        # -----------------------------------------------------

        if vectorstore is None:

            print(
                "   → Creating ChromaDB "
                "with first batch..."
            )

            vectorstore = Chroma.from_documents(
                documents=batch,
                embedding=embedding_model,
                persist_directory=persist_directory,
                collection_metadata={
                    "hnsw:space": "cosine"
                }
            )

        # -----------------------------------------------------
        # Remaining batches → add to existing DB
        # -----------------------------------------------------

        else:

            print(
                "   → Adding batch to existing ChromaDB..."
            )

            vectorstore.add_documents(batch)

        # -----------------------------------------------------
        # Update count
        # -----------------------------------------------------

        total_added += len(batch)

        print(
            f"   ✅ Batch {batch_number} completed"
        )

        print(
            f"   📊 Total documents added: "
            f"{total_added}"
        )

        # -----------------------------------------------------
        # Free current batch from RAM
        # -----------------------------------------------------

        del batch
        gc.collect()

    # ---------------------------------------------------------
    # 3. Verify
    # ---------------------------------------------------------

    if vectorstore is None:

        raise ValueError(
            "No documents were found to create "
            "the vector store."
        )

    print("\n" + "=" * 60)
    print("✅ Vector store creation completed")
    print(
        f"📊 Documents processed: {total_added}"
    )
    print(
        f"📁 Vector DB location: "
        f"{persist_directory}"
    )
    print("=" * 60)

    return vectorstore

db = create_vector_store(
    processed_file,
    persist_directory="dbv1/chroma_db",
    batch_size=25
)
import shutil

shutil.make_archive(
    "/content/chroma_db",
    "zip",
    "dbv1/chroma_db"
)

print("✅ ChromaDB zipped successfully")
from google.colab import files

files.download("/content/chroma_db.zip")

KeyboardInterrupt: 

Extracting the stored vector database

In [ ]:
# @title
import zipfile
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

with zipfile.ZipFile(
    "/content/chroma_db.zip",
    "r"
) as zip_ref:

    zip_ref.extractall("/content/dbv1/chroma_db")

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

db = Chroma(
    persist_directory="/content/dbv1/chroma_db",
    embedding_function=embedding_model
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
data = db.get()
print(data.keys())

In [ ]:
print(data["documents"][0])

Hybrid Retreiver Class

In [ ]:
# @title
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_community.retrievers import BM25Retriever
from pydantic import ConfigDict

class HybridRetriever(BaseRetriever):

    vector_retriever: object
    bm25_retriever: object
    k: int = 10
    rrf_k: int = 60

    model_config = ConfigDict(arbitrary_types_allowed=True)

    def _get_relevant_documents(self, query, *, run_manager=None):

        # 1. Retrieve using Vector Search
        vector_docs = self.vector_retriever.invoke(query)

        # 2. Retrieve using BM25
        bm25_docs = self.bm25_retriever.invoke(query)

        # 3. RRF Fusion
        scores = {}
        documents = {}

        # Vector results
        for rank, doc in enumerate(vector_docs, start=1):
            doc_id = doc.page_content

            scores[doc_id] = scores.get(doc_id, 0) + (
                1 / (self.rrf_k + rank)
            )

            documents[doc_id] = doc

        # BM25 results
        for rank, doc in enumerate(bm25_docs, start=1):
            doc_id = doc.page_content

            scores[doc_id] = scores.get(doc_id, 0) + (
                1 / (self.rrf_k + rank)
            )

            documents[doc_id] = doc

        # 4. Sort by RRF score
        ranked_docs = sorted(
            documents.items(),
            key=lambda x: scores[x[0]],
            reverse=True
        )

        # 5. Return top-k Documents
        return [
            doc
            for doc_id, doc in ranked_docs[:self.k]
        ]

# Get all documents from Chroma
all_docs = db.get()["documents"]

# Convert raw text into LangChain Documents
bm25_docs = [
    Document(page_content=text)
    for text in all_docs
]

# BM25 retriever
bm25_retriever = BM25Retriever.from_documents(bm25_docs)
bm25_retriever.k = 10


# Your existing vector retriever
vector_retriever = db.as_retriever(
    search_kwargs={"k": 10}
)


# Hybrid retriever
retriever = HybridRetriever(
    vector_retriever=vector_retriever,
    bm25_retriever=bm25_retriever,
    k=10
)

Normal Retreiver

In [ ]:
# @title
retriever = db.as_retriever(
    search_kwargs={"k": 10}
)

Reranker and Generate Func

In [ ]:
# @title
import json
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-base"
)


def get_raw_text(chunk):
    """
    Extract the raw text stored inside the chunk's original_content metadata.
    """

    if "original_content" not in chunk.metadata:
        return ""

    try:
        original_data = json.loads(
            chunk.metadata["original_content"]
        )

        return original_data.get("raw_text", "")

    except Exception as e:
        print(f"Error extracting raw text: {e}")
        return ""


def rerank_chunks(query, chunks, top_k=3):
    """
    Rerank retrieved chunks using their raw text.

    The reranker evaluates:
        (query, raw_text)

    rather than:
        (query, chunk.page_content)
    """

    if not chunks:
        return []

    # Extract raw text from every chunk
    raw_texts = [
        get_raw_text(chunk)
        for chunk in chunks
    ]

    # Create (query, document) pairs
    pairs = [
        (query, raw_text)
        for raw_text in raw_texts
    ]

    # Calculate relevance scores
    scores = reranker.predict(pairs)

    # Attach scores to chunks
    scored_chunks = list(
        zip(chunks, scores)
    )

    # Highest relevance first
    scored_chunks.sort(
        key=lambda x: x[1],
        reverse=True
    )

    # Return top-k chunks
    return [
        chunk
        for chunk, score in scored_chunks[:top_k]
    ]

def generate_final_answer(chunks, query):
    """Generate final answer using multimodal content"""

    chunks= rerank_chunks(query, chunks, top_k=3)

    try:
        # Build the text prompt
        prompt_text = f"""You are answering a question using documents retrieved from a knowledge base.

QUESTION:
{query}

The documents below have been ranked by a relevance model specifically for this question.
They are presented in descending order of relevance:
- Document 1 is the highest-ranked document.
- Later documents may be less relevant.
- Do not assume that every document is equally relevant.
- Prioritize information from the most relevant documents.
- Only use information that is actually supported by the retrieved documents.

RETRIEVED CONTEXT:
"""
        # chunks= chunks[:1]  # taking only the first chunk
        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"

            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])

                # Add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"

                # Add tables as HTML
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"

            prompt_text += "\n"

        prompt_text += """
ANSWERING INSTRUCTIONS:
- Use only information supported by the retrieved context.
- Do not introduce facts that are not present in the context.
- Prioritize the highest-ranked documents when they directly address the question.
- You may combine information from multiple documents when they clearly refer to the same subject.
- Do not combine unrelated passages merely because they were retrieved.
- If the retrieved context does not contain enough information to answer the question,
  explicitly state that the available context is insufficient.
- Give a concise, direct answer to the question.

ANSWER:
"""

        # Build message content starting with text
        # Build messages in Hugging Face chat format
        messages = [
            {
                "role": "user",
                "content": prompt_text

            }
        ]

        # Add images
        # for chunk in chunks:
        #     if "original_content" in chunk.metadata:
        #         original_data = json.loads(chunk.metadata["original_content"])
        #         images_base64 = original_data.get("images_base64", [])

        #         for image_base64 in images_base64:
        #             messages[0]["content"].append({
        #                 "type": "image",
        #                 "image": f"data:image/jpeg;base64,{image_base64}"
        #             })

        # Generate response
        response = llm(
            messages,
            max_new_tokens=500,
            do_sample=False
        )

        answer = response[0]["generated_text"][-1]["content"]

        return answer

    except Exception as e:
        print(f"❌ Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Loading Dataset and creating df

In [ ]:
!pip uninstall -y docx
!pip install -U python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.2 MB/s eta 0:00:00


In [ ]:
# @title
import os
import re
import pandas as pd
from docx import Document


def load_evaluation_dataset(file_path):
    """
    Parse the evaluation .docx file and convert it into a DataFrame.

    Expected format:

    Paper 1 — Paper Title

    Q1. Question
    Reference Answer: Answer
    Difficulty: Easy Source: Section 2

    Q2. Question
    Reference Answer: Answer
    Difficulty: Medium Source: Table 1
    """

    print("=" * 70)
    print("FUNCTION 1: Loading evaluation dataset")
    print("=" * 70)

    # ---------------------------------------------------------
    # 1. Check file
    # ---------------------------------------------------------
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    print(f"✓ Found file: {file_path}")

    # ---------------------------------------------------------
    # 2. Read DOCX
    # ---------------------------------------------------------
    extension = os.path.splitext(file_path)[1].lower()

    if extension != ".docx":
        raise ValueError("This function currently expects a .docx file.")

    doc = Document(file_path)

    # Extract non-empty paragraphs
    lines = [
        paragraph.text.strip()
        for paragraph in doc.paragraphs
        if paragraph.text.strip()
    ]

    print("✓ Document read successfully")
    print(f"✓ Non-empty paragraphs extracted: {len(lines)}")

    # ---------------------------------------------------------
    # 3. Patterns
    # ---------------------------------------------------------
    paper_pattern = re.compile(
        r"^Paper\s+(\d+)\s*[—-]\s*(.+)$"
    )

    question_pattern = re.compile(
        r"^Q(\d+)\.\s*(.+)$"
    )

    # ---------------------------------------------------------
    # 4. Find papers
    # ---------------------------------------------------------
    paper_indices = []

    for i, line in enumerate(lines):
        match = paper_pattern.match(line)

        if match:
            paper_indices.append(
                (i, match.group(1), match.group(2).strip())
            )

    if not paper_indices:
        raise ValueError(
            "No paper headings found. Expected format like:\n"
            "Paper 1 — Paper Title"
        )

    print(f"✓ Papers detected: {len(paper_indices)}")

    records = []

    # ---------------------------------------------------------
    # 5. Process each paper
    # ---------------------------------------------------------
    for paper_idx, (start_idx, paper_number, paper_name) in enumerate(
        paper_indices
    ):

        print("\n" + "-" * 70)
        print(f"Processing Paper {paper_number}: {paper_name}")
        print("-" * 70)

        # End of current paper
        if paper_idx + 1 < len(paper_indices):
            end_idx = paper_indices[paper_idx + 1][0]
        else:
            end_idx = len(lines)

        paper_lines = lines[start_idx + 1:end_idx]

        # -----------------------------------------------------
        # 6. Find questions
        # -----------------------------------------------------
        question_indices = []

        for i, line in enumerate(paper_lines):
            match = question_pattern.match(line)

            if match:
                question_indices.append(
                    (i, match.group(1), match.group(2).strip())
                )

        print(f"  ✓ Questions detected: {len(question_indices)}")

        # -----------------------------------------------------
        # 7. Process each question
        # -----------------------------------------------------
        for q_idx, (q_start, question_number, question) in enumerate(
            question_indices
        ):

            # Determine where this question ends
            if q_idx + 1 < len(question_indices):
                q_end = question_indices[q_idx + 1][0]
            else:
                q_end = len(paper_lines)

            question_lines = paper_lines[q_start + 1:q_end]

            # -------------------------------------------------
            # Find Reference Answer
            # -------------------------------------------------
            answer = None
            difficulty = None
            source = None

            for line in question_lines:

                # Reference Answer
                if line.startswith("Reference Answer:"):
                    answer = line[len("Reference Answer:"):].strip()

                # Difficulty + Source
                elif line.startswith("Difficulty:"):

                    difficulty_source = line[len("Difficulty:"):].strip()

                    # Extract difficulty
                    difficulty_match = re.match(
                        r"(Easy|Medium|Difficult)\s+Source:\s*(.*)",
                        difficulty_source
                    )

                    if difficulty_match:
                        difficulty = difficulty_match.group(1).strip()
                        source = difficulty_match.group(2).strip()

            # -------------------------------------------------
            # Validation
            # -------------------------------------------------
            if answer is None:
                print(
                    f"  ⚠ Warning: Missing answer for Q{question_number}"
                )

            if difficulty is None:
                print(
                    f"  ⚠ Warning: Missing difficulty for Q{question_number}"
                )

            if source is None:
                print(
                    f"  ⚠ Warning: Missing source for Q{question_number}"
                )

            # -------------------------------------------------
            # Add record
            # -------------------------------------------------
            records.append({
                "question": question,
                "answer": answer,
                "source": source,
                "Difficulty": difficulty,
                "paper_name": paper_name
            })

    # ---------------------------------------------------------
    # 8. Create DataFrame
    # ---------------------------------------------------------
    df = pd.DataFrame(records)

    print("\n" + "=" * 70)
    print("DATASET CREATED")
    print("=" * 70)

    print(f"Total questions: {len(df)}")
    print(f"Total papers: {df['paper_name'].nunique()}")

    print("\nQuestions per paper:")
    print(df.groupby("paper_name").size())

    # ---------------------------------------------------------
    # 9. Validation
    # ---------------------------------------------------------
    expected_questions_per_paper = 10

    counts = df.groupby("paper_name").size()

    if not all(counts == expected_questions_per_paper):
        print(
            "\n⚠ WARNING: Some papers do not contain exactly "
            "10 questions."
        )
    else:
        print("\n✓ Every paper contains exactly 10 questions.")

    # ---------------------------------------------------------
    # 10. Check for missing values
    # ---------------------------------------------------------
    print("\nMissing values:")
    print(df.isnull().sum())

    if not df.isnull().any().any():
        print("✓ No missing values detected.")

    # ---------------------------------------------------------
    # 11. Final column order
    # ---------------------------------------------------------
    df = df[
        [
            "question",
            "answer",
            "source",
            "Difficulty",
            "paper_name"
        ]
    ]

    print("\n✓ Final DataFrame ready.")
    print(f"✓ DataFrame shape: {df.shape}")

    return df

Generating Answers

In [ ]:
# @title
def generate_answers_for_dataset(df, retriever):
    """
    Runs the existing RAG pipeline for every question in the evaluation
    DataFrame.

    Existing pipeline:
        question
            ↓
        retriever.invoke(question)
            ↓
        generate_final_answer(chunks, question)
            ↓
        generated answer

    Adds:
        generated -> RAG-generated answer
        contexts  -> retrieved chunk contents

    Returns:
        A new DataFrame containing the generated answers and contexts.
    """

    print("=" * 70)
    print("FUNCTION 2: RUNNING RAG PIPELINE ON EVALUATION DATASET")
    print("=" * 70)

    result_df = df.copy()

    generated_answers = []
    retrieved_contexts = []

    total = len(result_df)

    print(f"Total questions to process: {total}")
    print()

    for idx, query in enumerate(result_df["question"]):

        question_number = idx + 1

        print("-" * 70)
        print(f"Question {question_number}/{total}")
        print(f"Query: {query}")

        try:
            # --------------------------------------------------
            # STEP 1: Retrieve relevant chunks
            # --------------------------------------------------
            print("\n[1/2] Retrieving relevant chunks...")

            chunks = retriever.invoke(query)

            print(f"✓ Retrieved {len(chunks)} chunks")

            # --------------------------------------------------
            # STEP 2: Generate final answer using your
            # existing multimodal generator
            # --------------------------------------------------
            print("[2/2] Generating answer...")

            generated_answer = generate_final_answer(
                chunks,
                query
            )

            print("✓ Answer generated successfully")

            # --------------------------------------------------
            # Convert retrieved chunks into text contexts
            # for RAGAs
            # --------------------------------------------------
            contexts = []

            for chunk in chunks:

                # LangChain Document
                if hasattr(chunk, "page_content"):
                    context = chunk.page_content

                # Plain string
                elif isinstance(chunk, str):
                    context = chunk

                # Dictionary
                elif isinstance(chunk, dict):
                    context = chunk.get(
                        "page_content",
                        chunk.get("content", str(chunk))
                    )

                else:
                    context = str(chunk)

                if context:
                    contexts.append(context)

            print(f"✓ Stored {len(contexts)} contexts")

            # --------------------------------------------------
            # Store results
            # --------------------------------------------------
            generated_answers.append(generated_answer)
            retrieved_contexts.append(contexts)

            # Short preview
            preview = str(generated_answer).replace("\n", " ")

            if len(preview) > 250:
                preview = preview[:250] + "..."

            print(f"\nGenerated answer:\n{preview}")

        except Exception as e:

            print("\n❌ ERROR")
            print(f"Question {question_number} failed.")
            print(f"Error type: {type(e).__name__}")
            print(f"Error message: {e}")

            # Keep DataFrame alignment intact
            generated_answers.append(None)
            retrieved_contexts.append([])

            print("⚠ Continuing with the next question...")

    # ----------------------------------------------------------
    # Add results to DataFrame
    # ----------------------------------------------------------
    result_df["generated"] = generated_answers
    result_df["contexts"] = retrieved_contexts

    # ----------------------------------------------------------
    # Final summary
    # ----------------------------------------------------------
    successful = result_df["generated"].notna().sum()
    failed = result_df["generated"].isna().sum()

    print("\n" + "=" * 70)
    print("RAG GENERATION COMPLETE")
    print("=" * 70)

    print(f"Total questions : {total}")
    print(f"Successful      : {successful}")
    print(f"Failed          : {failed}")

    if failed == 0:
        print("✓ All questions processed successfully!")
    else:
        print(f"⚠ {failed} question(s) failed.")
        print("  Check rows where 'generated' is None.")

    return result_df

Class for fixing RAGAs related Issue

In [ ]:
# @title
from __future__ import annotations

import asyncio
import json
import logging
import re
import threading
import typing as t

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

from ragas.llms.base import BaseRagasLLM
from langchain_core.outputs import LLMResult, Generation

logger = logging.getLogger("ragas_local_llm")


# ---------------------------------------------------------------------------
# 1. Custom exception -- failures must be loud, never silent
# ---------------------------------------------------------------------------
class JSONExtractionError(RuntimeError):
    """Raised when the model output cannot be safely turned into valid JSON.

    Carries the raw model output and (if any) the best-effort extracted
    candidate string, so you can inspect exactly what went wrong instead of
    getting a silent NaN with no context.
    """

    def __init__(self, message: str, raw_output: str, candidate: t.Optional[str] = None):
        super().__init__(message)
        self.raw_output = raw_output
        self.candidate = candidate

    def __str__(self) -> str:
        base = super().__str__()
        return (
            f"{base}\n"
            f"--- RAW MODEL OUTPUT (truncated to 1000 chars) ---\n"
            f"{self.raw_output[:1000]}\n"
            f"--- BEST-EFFORT CANDIDATE (truncated to 1000 chars) ---\n"
            f"{(self.candidate or '')[:1000]}"
        )


# ---------------------------------------------------------------------------
# 2. JSON extraction / repair pipeline
# ---------------------------------------------------------------------------
_CODE_FENCE_RE = re.compile(r"```(?:json|JSON)?\s*(.*?)```", re.DOTALL)


def _strip_code_fences(text: str) -> str:
    """If the text contains a ```json ... ``` (or plain ```) block, prefer its
    contents. Falls back to the original text if no fence is found."""
    match = _CODE_FENCE_RE.search(text)
    if match:
        return match.group(1).strip()
    return text


def _find_balanced_json_span(text: str) -> t.Optional[str]:
    """Scan for the first '{' or '[' and return the substring up to its
    matching closing bracket, respecting strings/escapes so braces inside
    quoted values don't confuse the matcher. Returns None if no balanced
    span is found.
    """
    start_idx = None
    for i, ch in enumerate(text):
        if ch in "{[":
            start_idx = i
            break
    if start_idx is None:
        return None

    open_ch = text[start_idx]
    close_ch = "}" if open_ch == "{" else "]"

    depth = 0
    in_string = False
    escape = False
    for i in range(start_idx, len(text)):
        ch = text[i]
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue
        else:
            if ch == '"':
                in_string = True
            elif ch in "{[":
                depth += 1
            elif ch in "}]":
                depth -= 1
                if depth == 0:
                    return text[start_idx : i + 1]
    # Unbalanced (model got cut off, etc.) -- no safe span found.
    return None


def _attempt_minor_repairs(candidate: str) -> t.Iterator[str]:
    """Yield progressively more aggressive, still-safe repair attempts.
    Each yielded string is tried with json.loads() by the caller; we do NOT
    fabricate content, only fix common, unambiguous formatting mistakes.
    """
    yield candidate  # try as-is first

    # Remove trailing commas before a closing bracket: {"a": 1,} -> {"a": 1}
    no_trailing_commas = re.sub(r",\s*([}\]])", r"\1", candidate)
    if no_trailing_commas != candidate:
        yield no_trailing_commas

    # Normalize "smart quotes" to plain double quotes (common with some
    # instruct models that stylize their punctuation).
    smart_quotes_fixed = (
        no_trailing_commas.replace("\u201c", '"')
        .replace("\u201d", '"')
        .replace("\u2018", "'")
        .replace("\u2019", "'")
    )
    if smart_quotes_fixed != no_trailing_commas:
        yield smart_quotes_fixed

    # Python-style single-quoted dict -> JSON double quotes. Only do this if
    # the candidate has no double quotes at all (i.e. it's unambiguous that
    # single quotes are being used as the string delimiter, not as an
    # apostrophe inside a double-quoted string).
    if '"' not in smart_quotes_fixed and "'" in smart_quotes_fixed:
        single_to_double = smart_quotes_fixed.replace("'", '"')
        yield single_to_double

    # Strip a stray trailing/leading backtick or comma the model sometimes
    # appends after the closing bracket.
    stripped_edges = smart_quotes_fixed.strip("`, \n\t")
    if stripped_edges != smart_quotes_fixed:
        yield stripped_edges


def extract_and_validate_json(raw_output: str) -> str:
    """Turn a raw, possibly messy model completion into a clean JSON string.

    Returns the *string* form of valid JSON (not the parsed object), because
    that's what RAGAs' RagasOutputParser expects to receive as generation
    text before it runs its own `model_validate_json()`.

    Raises JSONExtractionError if no safe repair produces valid JSON.
    """
    text = raw_output.strip()
    if not text:
        raise JSONExtractionError("Model returned an empty response.", raw_output)

    # Step 1: prefer fenced ```json blocks if present.
    fenced = _strip_code_fences(text)

    # Step 2: find the first balanced {...} or [...] span.
    candidate = _find_balanced_json_span(fenced)
    if candidate is None:
        # Maybe the fence stripping ate something useful -- retry on raw text.
        candidate = _find_balanced_json_span(text)

    if candidate is None:
        raise JSONExtractionError(
            "Could not locate a balanced JSON object/array in the model output.",
            raw_output,
        )

    # Step 3: try the candidate, then progressively safer repairs.
    last_error: t.Optional[Exception] = None
    for attempt in _attempt_minor_repairs(candidate):
        try:
            parsed = json.loads(attempt)
        except (json.JSONDecodeError, ValueError) as e:
            last_error = e
            continue
        # Re-serialize to guarantee canonical, strictly valid JSON text
        # (also collapses any repairs into a clean final string).
        return json.dumps(parsed)

    raise JSONExtractionError(
        f"Extracted a JSON-like span but could not parse it even after repairs: {last_error}",
        raw_output,
        candidate,
    )


def _strip_echoed_prompt(prompt_text: str, generated_text: str) -> str:
    """Some chat pipelines return the prompt + completion concatenated.
    If the generated text starts with (a prefix of) the prompt, remove it.
    """
    if generated_text.startswith(prompt_text):
        return generated_text[len(prompt_text) :].strip()
    # Also handle the common case where only the tail of the prompt
    # (e.g. after the last chat template turn) gets echoed.
    tail = prompt_text[-200:]
    idx = generated_text.find(tail)
    if idx != -1:
        return generated_text[idx + len(tail) :].strip()
    return generated_text.strip()


# ---------------------------------------------------------------------------
# 3. The RAGAs-compatible LLM wrapper
# ---------------------------------------------------------------------------
class RobustQwenRagasLLM(BaseRagasLLM):
    """A `BaseRagasLLM` implementation backed by a local HF `transformers`
    text-generation pipeline, with robust JSON extraction/repair applied to
    every generation before RAGAs sees it.

    Pass this directly as `llm=` to `ragas.evaluate()`.
    """

    def __init__(
        self,
        model_name: str = "Qwen/Qwen2.5-1.5B-Instruct",
        max_new_tokens: int = 512,
        device_map: str = "auto",
        debug: bool = False,
        max_repair_retries: int = 1,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        self.debug = debug
        self.max_repair_retries = max_repair_retries

        # We do NOT set `multiple_completion_supported = True`: some
        # metrics (e.g. answer_relevancy with strictness>1) ask for n>1
        # completions. We support it by looping locally (see generate_text),
        # but we don't advertise native batched sampling since HF pipelines
        # don't give us that for free without extra complexity.
        self.multiple_completion_supported = False

        logger.info("Loading tokenizer/model: %s", model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map=device_map,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        )

        # Avoid the "Both max_new_tokens and max_length" warning: some base
        # generation configs ship with a default max_length that conflicts
        # with max_new_tokens. We neutralize it here instead of passing both.
        self.model.generation_config.max_length = None
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

        self.pipe = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
        )

        # transformers pipelines / the underlying CUDA context are not
        # guaranteed safe for concurrent calls from multiple threads at once.
        # RAGAs' executor can fire several async tasks concurrently even on
        # a single machine, so we serialize actual model calls with a lock.
        # (Set RunConfig(max_workers=1) on evaluate() as well for good measure.)
        self._inference_lock = threading.Lock()

    # -- required by BaseRagasLLM -----------------------------------------
    def is_finished(self, response: LLMResult) -> bool:
        try:
            for gen_list in response.generations:
                for gen in gen_list:
                    if not gen.text or not gen.text.strip():
                        return False
            return True
        except Exception:
            return False

    def generate_text(
        self,
        prompt: t.Any,  # PromptValue-like: has .to_string() or .text
        n: int = 1,
        temperature: t.Optional[float] = 0.01,
        stop: t.Optional[t.List[str]] = None,
        callbacks: t.Any = None,
    ) -> LLMResult:
        prompt_str = self._prompt_to_string(prompt)

        completions: t.List[str] = []
        for i in range(max(n, 1)):
            cleaned_json_text = self._generate_one(prompt_str, call_index=i)
            completions.append(cleaned_json_text)

        # Single logical prompt -> one outer list containing all n completions,
        # matching how RAGAs' RagasOutputParser reads `resp.generations[0][i].text`.
        return LLMResult(generations=[[Generation(text=c) for c in completions]])

    async def agenerate_text(
        self,
        prompt: t.Any,
        n: int = 1,
        temperature: t.Optional[float] = 0.01,
        stop: t.Optional[t.List[str]] = None,
        callbacks: t.Any = None,
    ) -> LLMResult:
        # HF pipeline inference is synchronous/blocking; run it in a worker
        # thread so we don't block the event loop RAGAs uses for concurrency.
        return await asyncio.to_thread(
            self.generate_text, prompt, n, temperature, stop, callbacks
        )

    # -- internals -----------------------------------------------------------
    @staticmethod
    def _prompt_to_string(prompt: t.Any) -> str:
        if hasattr(prompt, "to_string"):
            return prompt.to_string()
        if hasattr(prompt, "text"):
            return prompt.text
        return str(prompt)

    def _generate_one(self, prompt_str: str, call_index: int = 0) -> str:
        """Run the model once and return a clean, validated JSON string.
        Raises JSONExtractionError if it cannot be safely recovered even
        after a bounded number of regeneration attempts.
        """
        messages = [{"role": "user", "content": prompt_str}]

        last_exc: t.Optional[JSONExtractionError] = None
        for attempt in range(self.max_repair_retries + 1):
            with self._inference_lock:
                outputs = self.pipe(
                    messages,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=False,       # deterministic
                    temperature=None,      # must be None when do_sample=False
                    top_p=None,
                    top_k=None,
                    repetition_penalty=1.05,
                    pad_token_id=self.tokenizer.pad_token_id,
                    return_full_text=False,
                )

            raw_text = outputs[0]["generated_text"]
            if isinstance(raw_text, list):
                # Some pipeline versions return chat-formatted output; grab
                # the last assistant turn's content.
                raw_text = raw_text[-1].get("content", "")

            raw_text = _strip_echoed_prompt(prompt_str, raw_text)

            if self.debug:
                print(f"\n=== RAW MODEL OUTPUT (call {call_index}, attempt {attempt}) ===")
                print(raw_text)

            try:
                clean_json = extract_and_validate_json(raw_text)
            except JSONExtractionError as e:
                last_exc = e
                if self.debug:
                    print("=== EXTRACTED JSON ===\n(failed to extract)")
                    print(f"=== ERROR ===\n{e}")
                continue  # retry generation, if attempts remain

            if self.debug:
                print("=== EXTRACTED / VALIDATED JSON ===")
                print(clean_json)
                print("=== PARSED OUTPUT ===")
                print(json.loads(clean_json))

            return clean_json

        # Exhausted retries -- fail loudly, do not fabricate a result.
        assert last_exc is not None
        raise last_exc

RAGAs Evaluation (returns df)

In [ ]:
# @title
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall
)

def evaluate_with_ragas(df, evaluator_llm, evaluator_embeddings):
    """
    Evaluate the RAG pipeline using RAGAs.

    Metrics:
        1. Faithfulness
        2. Answer Relevancy
        3. Context Recall
    """

    print("=" * 70)
    print("FUNCTION 3: RAGAs Evaluation")
    print("=" * 70)

    # ---------------------------------------------------------
    # 1. Check required columns
    # ---------------------------------------------------------

    required_columns = [
        "question",
        "answer",
        "generated",
        "contexts"
    ]

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )

    print("✓ Required columns found")

    # ---------------------------------------------------------
    # 2. Remove rows where generation failed
    # ---------------------------------------------------------

    evaluation_df = df.dropna(
        subset=["question", "answer", "generated", "contexts"]
    ).copy()

    print(
        f"✓ Evaluating {len(evaluation_df)} "
        f"of {len(df)} questions"
    )

    # ---------------------------------------------------------
    # 3. Clean the data types
    # ---------------------------------------------------------

    # Question, reference answer and generated answer
    # must be strings.
    evaluation_df["question"] = (
        evaluation_df["question"]
        .astype(str)
        .str.strip()
    )

    evaluation_df["answer"] = (
        evaluation_df["answer"]
        .astype(str)
        .str.strip()
    )

    evaluation_df["generated"] = (
        evaluation_df["generated"]
        .astype(str)
        .str.strip()
    )

    # retrieved_contexts must be a LIST of strings
    def clean_contexts(contexts):

        if contexts is None:
            return []

        # If a single string was stored instead of a list
        if isinstance(contexts, str):
            return [contexts]

        # Otherwise convert every context to string
        return [
            str(context).strip()
            for context in contexts
            if context is not None
        ]

    evaluation_df["contexts"] = (
        evaluation_df["contexts"]
        .apply(clean_contexts)
    )

    # Remove rows with no retrieved context
    evaluation_df = evaluation_df[
        evaluation_df["contexts"].apply(len) > 0
    ].copy()

    print(
        f"✓ {len(evaluation_df)} valid rows after data cleaning"
    )

    # ---------------------------------------------------------
    # 4. Create exactly the columns expected by RAGAs
    # ---------------------------------------------------------

    ragas_df = evaluation_df[
        [
            "question",
            "answer",
            "generated",
            "contexts"
        ]
    ].rename(
        columns={
            "question": "user_input",
            "answer": "reference",
            "generated": "response",
            "contexts": "retrieved_contexts"
        }
    )

    # ---------------------------------------------------------
    # 5. Convert to HuggingFace Dataset
    # ---------------------------------------------------------

    from datasets import Dataset

    ragas_dataset = Dataset.from_pandas(
        ragas_df,
        preserve_index=False
    )

    print("✓ RAGAs dataset created")

    # ---------------------------------------------------------
    # 6. Configure RAGAs metrics
    # ---------------------------------------------------------

    faithfulness.llm = evaluator_llm

    answer_relevancy.llm = evaluator_llm
    answer_relevancy.embeddings = evaluator_embeddings

    context_recall.llm = evaluator_llm

    metrics = [
        faithfulness,
        answer_relevancy,
        context_recall
    ]

    print("✓ RAGAs metrics initialized")
    print("  - Faithfulness")
    print("  - Answer Relevancy")
    print("  - Context Recall")

    # ---------------------------------------------------------
    # 7. Run evaluation
    # ---------------------------------------------------------

    print("\nStarting RAGAs evaluation...")
    print("This may take some time because RAGAs uses LLM calls.")
    print()

    run_config = RunConfig(
    timeout=600,
    max_workers=1,
      )

    result= evaluate(
        dataset=ragas_dataset,
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        metrics=[faithfulness, answer_relevancy, context_recall],
        # Serialize execution: safer for a single local GPU model instance.
        run_config=run_config,
        raise_exceptions=False,  # surface JSONExtractionError instead of hiding it as NaN
    )

    # ---------------------------------------------------------
    # 8. Convert result to DataFrame
    # ---------------------------------------------------------

    results_df = result.to_pandas()

    print("\n" + "=" * 70)
    print("RAGAs EVALUATION COMPLETE")
    print("=" * 70)

    print("\nAverage scores:")
    print(results_df.mean(numeric_only=True))

    return results_df

Create llm object

In [ ]:
# @title
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
df_qs= load_evaluation_dataset("/content/evaluation_dataset.docx")
df_qs.head()
df_with_answers= generate_answers_for_dataset(df_qs, retriever)

FUNCTION 1: Loading evaluation dataset
✓ Found file: /content/evaluation_dataset.docx
✓ Document read successfully
✓ Non-empty paragraphs extracted: 155
✓ Papers detected: 5

----------------------------------------------------------------------
Processing Paper 1: One Weird Trick for Parallelizing CNNs (Krizhevsky, 2014)
----------------------------------------------------------------------
  ✓ Questions detected: 10

----------------------------------------------------------------------
Processing Paper 2: Very Deep Convolutional Networks for Large-Scale Image Recognition (VGG)
----------------------------------------------------------------------
  ✓ Questions detected: 10

----------------------------------------------------------------------
Processing Paper 3: Deep Residual Learning for Image Recognition (ResNet)
----------------------------------------------------------------------
  ✓ Questions detected: 10

----------------------------------------------------------------------

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


✓ Answer generated successfully
✓ Stored 10 contexts

Generated answer:
The core parallelization strategy proposed in the documents for training convolutional neural networks across multiple GPUs is data parallelism in the convolutional layers and model parallelism in the fully-connected layers. This is illustrated in Fi...
----------------------------------------------------------------------
Question 2/50
Query: Which dataset was used to evaluate the proposed parallelization scheme, and how large is it?

[1/2] Retrieving relevant chunks...
✓ Retrieved 10 chunks
[2/2] Generating answer...


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ Answer generated successfully
✓ Stored 10 contexts

Generated answer:
The question cannot be answered based solely on the provided context. The documents do not mention any specific dataset used for evaluating the proposed parallelization scheme or provide information about the size of such a dataset.
----------------------------------------------------------------------
Question 3/50
Query: What hardware configuration did the author use to run the experiments?

[1/2] Retrieving relevant chunks...
✓ Retrieved 10 chunks
[2/2] Generating answer...


...

✓ Answer generated successfully
✓ Stored 10 contexts

Generated answer:
In Table 2(f) of the provided context, FPN's Fast R-CNN (2fc head) shows an AP improvement of 0.3 compared to a baseline using a comparable 2fc head with single-scale C5 features. This comparison is considered fairer than comparing against the conv5-...
----------------------------------------------------------------------
Question 49/50
Query: How did FPN's instance segmentation proposal method compare to DeepMask and SharpMask in terms of accuracy (AR) and speed?

[1/2] Retrieving relevant chunks...
✓ Retrieved 10 chunks
[2/2] Generating answer...


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ Answer generated successfully
✓ Stored 10 contexts

Generated answer:
The documents provided do not contain specific information about comparing FPN's instance segmentation proposal method with DeepMask and SharpMask in terms of accuracy (AR) and speed. Therefore, the available context is insufficient to answer the que...
----------------------------------------------------------------------
Question 50/50
Query: What does the ablation using only the finest pyramid level P2 (Table 1, row f) reveal about the role of the pyramid structure versus simply increasing the number of anchors?

[1/2] Retrieving relevant chunks...
✓ Retrieved 10 chunks
[2/2] Generating answer...


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ Answer generated successfully
✓ Stored 10 contexts

Generated answer:
The ablation using only the finest pyramid level P2 reveals that a larger number of anchors alone is not sufficient to improve accuracy. Additionally, it shows that RoI pooling, which is less sensitive to region scales, can lead to marginal degradati...

RAG GENERATION COMPLETE
Total questions : 50
Successful      : 50
Failed          : 0
✓ All questions processed successfully!


In [ ]:
from google.colab import files
df_with_answers.to_csv('df_with_answers.csv')
files.download('df_with_answers.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

RAGAs compatibility Class & Final Call

In [ ]:
# @title
from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.metrics import faithfulness, answer_relevancy, context_recall
from langchain_huggingface import HuggingFaceEmbeddings
import pandas as pd
import ast
# Assumes these already exist in your environment, per your setup:
#   ragas_dataset
#   evaluator_embeddings

evaluator_llm = RobustQwenRagasLLM(
        model_name="Qwen/Qwen2.5-1.5B-Instruct",
        max_new_tokens=1024,
        debug=True,          # flip to False once you trust it
        max_repair_retries=1,  # regenerate once more if JSON extraction fails
    )

evaluator_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

df_with_answers = pd.read_csv("/content/df_with_answers.csv")

df_with_answers["contexts"] = df_with_answers["contexts"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

df_with_answers= df_with_answers.iloc[:40]
results_df = evaluate_with_ragas(
    df_with_answers,
    evaluator_llm,
    evaluator_embeddings
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...


=== RAW MODEL OUTPUT (call 0, attempt 0) ===
{
    "question": "What did the authors propose about the auxiliary classifiers?",
    "noncommittal": 1
}
=== EXTRACTED / VALIDATED JSON ===
{"question": "What did the authors propose about the auxiliary classifiers?", "noncommittal": 1}
=== PARSED OUTPUT ===
{'question': 'What did the authors propose about the auxiliary classifiers?', 'noncommittal': 1}


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== RAW MODEL OUTPUT (call 1, attempt 0) ===
{
    "question": "What did the authors propose about the auxiliary classifiers?",
    "noncommittal": 1
}
=== EXTRACTED / VALIDATED JSON ===
{"question": "What did the authors propose about the auxiliary classifiers?", "noncommittal": 1}
=== PARSED OUTPUT ===
{'question': 'What did the authors propose about the auxiliary classifiers?', 'noncommittal': 1}

=== RAW MODEL OUTPUT (call 2, attempt 0) ===
{
    "question": "What did the authors propose about the auxiliary classifiers?",
    "noncommittal": 1
}
=== EXTRACTED / VALIDATED JSON ===
{"question": "What did the authors propose about the auxiliary classifiers?", "noncommittal": 1}
=== PARSED OUTPUT ===
{'question': 'What did the authors propose about the auxiliary classifiers?', 'noncommittal': 1}


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


RAGAs EVALUATION COMPLETE

Average scores:
faithfulness        0.333333
answer_relevancy    0.524411
context_recall      1.000000
dtype: float64


In [ ]:
# @title
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
import pandas as pd
import ast

# Small local LLM for RAGAs evaluation
evaluator_pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_new_tokens=256,
    temperature=0.1,
    device_map="auto"
)

evaluator_llm = HuggingFacePipeline(
    pipeline=evaluator_pipe
)

# Free local embedding model


results_df = evaluate_with_ragas(
    df_with_answers,
    evaluator_llm,
    evaluator_embeddings
)
results_df= evaluate_with_ragas(df_with_answers, evaluator_llm, evaluator_embeddings)

In [ ]:
from google.colab import files
results_df.to_csv("results_df.csv")
files.download('results_df.csv')

In [ ]:
import pandas as pd
df= pd.read_csv("/content/results_df.csv")

In [ ]:
df.columns

In [ ]:
df['faithfulness'][0]

In [ ]:
'''The dataset used to evaluate the proposed parallelization scheme is the ILSVRC-2012 dataset.
 It includes images of 1000 classes, and is split into three sets: training (1.3M images), validation
 (50K images), and testing (100K images with held-out class labels).'''